# Advanced Retrievers — Multi-Query, Self-Querying, and Re-ranking

`Build_Smarter_AI_Apps` (Section 8) and `vector_databases_for_rag` already gave you the retrieval toolkit this notebook builds on: `Document`/`RecursiveCharacterTextSplitter` mechanics, `Chroma` + `HuggingFaceEmbeddings` fundamentals, plain `.as_retriever()`, metadata filtering via a hand-written `filter=` dict, and the two-chunk-size trade-off behind `ParentDocumentRetriever`. This notebook assumes all of that and doesn't re-teach it.

Three specific weaknesses of plain similarity search remain, and each gets its own fix here:

1. **Vocabulary / facet mismatch** — a compound question bundles several sub-questions into one embedding, so a real, relevant chunk can get missed even though it genuinely exists in your knowledge base. **`MultiQueryRetriever`** fixes this by asking an LLM to generate several rephrasings of the question, retrieving for each, and taking the union.
2. **The query secretly contains a filter** — "What are the critical IT policies from 2025?" is really two things smashed together: a semantic search ("policies") and a structured filter (`department=IT`, `priority=critical`, `year=2025`). In `vector_databases_for_rag` you wrote that filter dict by hand. **`SelfQueryRetriever`** has an LLM write it for you, straight from the natural-language question.
3. **The candidate pool is right but the order isn't** — a genuinely relevant document can be buried at position 3 behind two documents that only share surface vocabulary with the query. **Re-ranking** fixes the *order* of an already-retrieved pool using a cross-encoder — a different, complementary problem from the first two (this closes a loose thread `vector_databases_for_rag` §9 named as an exercise but never built).

**A packaging note before we start:** `langchain-community` isn't installed in this project's `upskill2` environment (same reason `Build_Smarter_AI_Apps` moved `TextLoader` and `ParentDocumentRetriever` onto `langchain_classic`) — and `SelfQueryRetriever`'s built-in Chroma translator happens to live inside `langchain_community.query_constructors.chroma`. Rather than pull that whole package back in for one class, Section 4 below writes the ~20-line translator by hand instead. Verified directly: `langchain_classic.retrievers.self_query.base._get_builtin_translator` raises `ImportError` on this machine without `langchain-community` installed — this isn't a guess, and nothing gets added to `requirements.txt` to work around it.

**On execution:** every cell that doesn't require your Anthropic API key was actually run while building this notebook, and the output below is real. Cells that need your key are marked **⚠️ needs your API key** and were left for you to run — same reasoning as the setup cell using `getpass` instead of a pasted key. (Section 6's re-ranking cells need no API key at all — the cross-encoder is a separate, local, non-Anthropic model — so those are fully executed, not just the construction halves.)

## Setup

In [1]:
# pip install -U langchain langchain-anthropic langchain-classic langchain-chroma langchain-huggingface langgraph

import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
print("✅ API key set.")

from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings


def get_llm(max_tokens=512, temperature=0):
    """Factory function, same pattern as 01/02 -- lets different sections use
    different temperature/max_tokens without sharing one global object."""
    return ChatAnthropic(model="claude-haiku-4-5", max_tokens=max_tokens, temperature=temperature)


print("Imports OK")

✅ API key set.


/Users/Sarah/venvs/upskill2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. The knowledge base — a policy set with real metadata to filter on

`vector_databases_for_rag` Section 5 filtered on a single `category` field. Section 4 of this notebook (`SelfQueryRetriever`) needs an LLM to reliably extract *several* different filterable attributes from a sentence, so the knowledge base here is deliberately wider than 02's HR/IT split: 10 policies across 4 departments, 3 years, and 2 priority levels. This also gives Section 3 (`MultiQueryRetriever`) a real compound-question case to work with — a 2-topic knowledge base can't demonstrate a retriever whose whole point is bridging *several* topics in one query.

In [2]:
policies = [
    ("Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval. "
     "Requests must be submitted at least 48 hours in advance through the HR portal.",
     {"department": "HR", "year": 2024, "priority": "standard"}),
    ("Vacation Policy: Full-time employees accrue 15 vacation days per year, accrued monthly. "
     "Unused vacation days roll over up to a maximum of 5 days into the following year.",
     {"department": "HR", "year": 2023, "priority": "standard"}),
    ("Parental Leave Policy: Employees are eligible for 12 weeks of paid leave following the birth, "
     "adoption, or fostering of a child, regardless of gender.",
     {"department": "HR", "year": 2025, "priority": "standard"}),
    ("Mobile Phone Policy: Company-issued phones may be used for personal calls within reason. "
     "Employees must report a lost or stolen device to IT within 24 hours.",
     {"department": "IT", "year": 2023, "priority": "standard"}),
    ("Expense Policy: Business expenses under $50 do not require pre-approval but must be submitted "
     "with receipts within 30 days. Expenses over $50 require manager sign-off before purchase.",
     {"department": "Finance", "year": 2024, "priority": "standard"}),
    ("Data Security Policy: All customer data must be encrypted at rest and in transit. Any suspected "
     "breach must be reported to the security team within 1 hour of discovery.",
     {"department": "IT", "year": 2025, "priority": "critical"}),
    ("Password Policy: Passwords must be at least 14 characters and rotated every 90 days. "
     "Multi-factor authentication is mandatory for all systems handling customer data.",
     {"department": "IT", "year": 2025, "priority": "critical"}),
    ("Travel and Reimbursement Policy: Employees traveling for business must book through the approved "
     "travel portal. Reimbursement requests must be filed within 14 days of return.",
     {"department": "Finance", "year": 2025, "priority": "standard"}),
    ("Code of Conduct: Employees must avoid conflicts of interest and disclose any outside business "
     "relationships that could affect their judgment at the company.",
     {"department": "Legal", "year": 2023, "priority": "critical"}),
    ("Non-Disclosure Policy: Employees must not share confidential company information with third "
     "parties without prior written approval from the Legal department.",
     {"department": "Legal", "year": 2024, "priority": "critical"}),
]

docs = [Document(page_content=text, metadata=meta) for text, meta in policies]

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embedding_model, collection_name="advanced_retrievers_policies")
print(f"Indexed {len(docs)} documents.")

Indexed 10 documents.


### 🔗 Ties back to theory — nothing new here mechanically

`Chroma.from_documents(docs, embedding_model)` is the exact same call dissected in `Build_Smarter_AI_Apps` Section 7 and `vector_databases_for_rag` Section 5 — embed each document once with the sentence-transformer, store `(vector, page_content, metadata)` triples in an HNSW-indexed collection. The only thing that changed is the metadata now carries three real, LLM-describable attributes (`department`, `year`, `priority`) instead of one.

## 2. The problem: a single embedding can't represent a compound question

Everything downstream depends on a real limitation, so let's see it happen before fixing it. Ask a single question that genuinely spans three unrelated policies at once — remote work, data security, and expenses.

In [5]:
compound_question = (
    "What do I need to know about working from home, keeping my laptop secure, "
    "and expensing my home office chair?"
)

print("Plain similarity_search, k=3:")
for d in vectorstore.similarity_search(compound_question, k=3):
    print(f"  [{d.metadata['department']}] {d.page_content[:70]}...")

print("\nSame question, k=5 (checking whether the missing angle just ranks lower):")
for rank, d in enumerate(vectorstore.similarity_search(compound_question, k=5), 1):
    print(f"  {rank}. [{d.metadata['department']}] {d.page_content[:70]}...")

Plain similarity_search, k=3:
  [HR] Remote Work Policy: Employees may work remotely up to 3 days per week ...
  [IT] Mobile Phone Policy: Company-issued phones may be used for personal ca...
  [IT] Data Security Policy: All customer data must be encrypted at rest and ...

Same question, k=5 (checking whether the missing angle just ranks lower):
  1. [HR] Remote Work Policy: Employees may work remotely up to 3 days per week ...
  2. [IT] Mobile Phone Policy: Company-issued phones may be used for personal ca...
  3. [IT] Data Security Policy: All customer data must be encrypted at rest and ...
  4. [IT] Password Policy: Passwords must be at least 14 characters and rotated ...
  5. [HR] Vacation Policy: Full-time employees accrue 15 vacation days per year,...


**Real result, worth reading carefully:** the Expense Policy — genuinely the right document for "expensing my home office chair" — doesn't appear even at `k=5`. It isn't ranked low; it's simply not among the 5 closest vectors to this question's single embedding, because that embedding ends up somewhere between "remote work" and "security," nowhere near "expenses." This is an honest result on a small, clean, 10-document corpus with a strong embedding model — with a larger or jargon-heavier corpus this effect gets worse, not better.

Worth being equally honest in the other direction: on *this* small corpus, plain search already handles most single-topic questions correctly (try re-running the cell above with a non-compound question like `"How many vacation days do I get?"` and it lands squarely on the right document at `k=1`). `MultiQueryRetriever`'s real value here is specifically the compound-question case, not "make every search better."

## 3. `MultiQueryRetriever` — casting a wider net across facets

The fix: instead of trusting one embedding of the raw question, ask an LLM to generate several *different phrasings* of the same question, retrieve separately for each, and union the results (de-duplicated). Each rephrasing is a different point in embedding space — one phrasing is more likely to land near "remote work," another near "security," another near "expenses," even though the single original phrasing landed near none of them precisely enough.

**⚠️ Needs your API key to run — the cell below calls the LLM twice (once per rephrasing round, once per generated query's retrieval isn't an LLM call, but the rephrasing step itself is).**

In [6]:
import logging

logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=get_llm(),
)

results = multi_query_retriever.invoke(compound_question)
print(f"MultiQueryRetriever returned {len(results)} unique documents:")
for d in results:
    print(f"  [{d.metadata['department']}] {d.page_content[:70]}...")

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['# Alternative Question Versions', '1. How can I secure my laptop while working remotely and what are the best practices for home office cybersecurity?', '2. What home office equipment and furniture expenses are tax-deductible, and how do I properly expense items like office chairs?', '3. What are the key considerations for setting up a productive and secure home workspace, including device protection and cost management?']


MultiQueryRetriever returned 6 unique documents:
  [Legal] Non-Disclosure Policy: Employees must not share confidential company i...
  [IT] Password Policy: Passwords must be at least 14 characters and rotated ...
  [HR] Remote Work Policy: Employees may work remotely up to 3 days per week ...
  [IT] Data Security Policy: All customer data must be encrypted at rest and ...
  [Finance] Expense Policy: Business expenses under $50 do not require pre-approva...
  [Finance] Travel and Reimbursement Policy: Employees traveling for business must...


### 🔍 Under the hood — what `MultiQueryRetriever.from_llm` actually builds

`from_llm(retriever=base_retriever, llm=...)` with no `prompt=` argument uses this exact built-in default (verified by inspecting the installed package directly, not assumed):

```text
You are an AI language model assistant. Your task is
    to generate 3 different versions of the given user
    question to retrieve relevant documents from a vector  database.
    By generating multiple perspectives on the user question,
    your goal is to help the user overcome some of the limitations
    of distance-based similarity search. Provide these alternative
    questions separated by newlines. Original question: {question}
```

Step by step, one `.invoke(compound_question)` call does:
1. Formats the prompt above with your question, sends **one** LLM call, gets back 3 newline-separated rephrasings.
2. `LineListOutputParser.parse(...)` splits the response on `"\n"` and drops empty lines — nothing fancier than `text.strip().split("\n")`.
3. Calls `base_retriever.invoke(...)` once **per rephrasing** (3 separate retrieval calls here, `k=2` each — up to 6 documents before dedup).
4. Merges all results into a single list, removing exact duplicates (same `page_content` + `metadata`) — so the final count is often less than `3 × k`, and depends entirely on how much the rephrasings' top-k results overlap.

Total cost of that one `.invoke()`: 1 LLM call + 3 retrieval calls, versus 1 LLM call + 1 retrieval call for a plain retriever wrapped the same way — worth knowing before reaching for this on a latency-sensitive path.

**Q: so it doesnt actually include the LLM generation step at the end?**

Correct — and worth being precise about why. `MultiQueryRetriever` is a **retriever**, and every retriever's `.invoke()` returns `list[Document]`, full stop — that's the interface, the same one `vectorstore.as_retriever()` implements. Look at the printed output two cells up: it's raw policy text, not a synthesized sentence answering the compound question. The LLM gets used *inside* `.invoke()` only for step 1 above (generating the 3 rephrasings) — never to read the retrieved documents and write an answer from them.

Turning those retrieved documents into an actual answer is a separate, later step you'd add on top — either wrapping the retriever as a tool for `create_agent` (`Build_Smarter_AI_Apps` §9's pattern), or a plain prompt-and-generate call like `04_evaluation_for_llm_apps`'s `generate_rag_answer` does. Retrieval and generation stay two separate operations here; `MultiQueryRetriever` only ever touches the first one.

### 🔗 Ties back to theory — this is the embedding-space geometry from `vector_databases_for_rag`, applied deliberately

Section 1 and 3 of `vector_databases_for_rag` established that a query is just one point in 384-dimensional space, and "relevant" means "nearby." A compound question's single embedding is, geometrically, some kind of average of its sub-topics — which can end up close to *none* of them individually (exactly what Section 2 above just demonstrated with the missing Expense Policy). Multiple rephrasings are multiple different points sampled around that same underlying intent; the union recovers coverage a single point structurally cannot.

## 4. `SelfQueryRetriever` — letting an LLM write the `filter=` dict for you

`vector_databases_for_rag` Section 5 filtered like this: `vectorstore.similarity_search(query, filter={"category": "landmarks"})` — you had to already know the exact field name and value. `SelfQueryRetriever` removes that requirement: describe the available metadata fields once (via `AttributeInfo`), and an LLM parses a plain question like *"What are the critical IT policies?"* into a semantic query (`"policies"`) plus a structured filter (`department="IT"`, `priority="critical"`) automatically.

### 🔍 Under the hood — what `$eq` and `$and` actually are

Before reading `SimpleChromaTranslator` below, worth knowing what its output — dicts like `{"department": {"$eq": "IT"}}` — actually means. Chroma's `where` filter syntax borrows a convention from MongoDB's query language: a key starting with `$` is an **operator**, not a field name, and the `$`-prefix exists specifically to disambiguate "this is an instruction" from "this is a literal value." `{"department": {"$eq": "IT"}}` reads as *"match documents where the `department` field is equal to `'IT'`"* — `$eq` is just the operator name for "equals." `{"$and": [{"department": {"$eq": "IT"}}, {"priority": {"$eq": "critical"}}]}` reads as *"match documents where both of these are true at once"* — `$and` combines multiple conditions.

Chroma supports the same small set of operators `SimpleChromaTranslator._cmp_map` below translates to: `$eq`/`$ne` (equal/not equal), `$gt`/`$gte`/`$lt`/`$lte` (greater/less than, for numbers), `$in`/`$nin` (value is/isn't in a list), plus `$and`/`$or` for combining conditions. Every method on `SimpleChromaTranslator` is doing exactly one job: look up which `$`-operator corresponds to LangChain's internal `Comparator`/`Operator` enum value, and build the dict shape above.

In [11]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.structured_query import (
    Comparator,
    Operator,
    Comparison,
    Operation,
    StructuredQuery,
    Visitor,
)


class SimpleChromaTranslator(Visitor):
    """Hand-written stand-in for langchain_community's ChromaTranslator --
    same Visitor interface SelfQueryRetriever expects, zero langchain_community
    dependency. Chroma's `where` filter syntax uses Mongo-style operator keys
    ($eq, $and, ...), which is all this class maps onto."""

    allowed_comparators = (
        Comparator.EQ, Comparator.NE, Comparator.GT, Comparator.GTE,
        Comparator.LT, Comparator.LTE, Comparator.IN, Comparator.NIN,
    )
    allowed_operators = (Operator.AND, Operator.OR)

    _op_map = {Operator.AND: "$and", Operator.OR: "$or"}
    _cmp_map = {
        Comparator.EQ: "$eq", Comparator.NE: "$ne", Comparator.GT: "$gt",
        Comparator.GTE: "$gte", Comparator.LT: "$lt", Comparator.LTE: "$lte",
        Comparator.IN: "$in", Comparator.NIN: "$nin",
    }

    # write a brief docstring on what each of these do
    def visit_operation(self, operation: Operation) -> dict:
        args = [arg.accept(self) for arg in operation.arguments]
        return {self._op_map[operation.operator]: args}

    def visit_comparison(self, comparison: Comparison) -> dict:
        return {comparison.attribute: {self._cmp_map[comparison.comparator]: comparison.value}}

    def visit_structured_query(self, structured_query: StructuredQuery):
        # I believe ethis is the one that we use
        if structured_query.filter is None:
            return structured_query.query, {}
        return structured_query.query, {"filter": structured_query.filter.accept(self)}


# Proof the translator is wired correctly, with no LLM involved yet: build the
# tree by hand, exactly as if an LLM had already parsed
# "critical IT policies from 2025" into it.
_translator = SimpleChromaTranslator()
_manual_tree = Operation(
    operator=Operator.AND,
    arguments=[
        Comparison(comparator=Comparator.EQ, attribute="department", value="IT"),
        Comparison(comparator=Comparator.EQ, attribute="priority", value="critical"),
        Comparison(comparator=Comparator.EQ, attribute="year", value=2025),
    ],
)
_query_text, _kwargs = _translator.visit_structured_query(
    # use a structured query type to test. Note this is the type thats is returned when you parse i think?
    StructuredQuery(query="policies", filter=_manual_tree, limit=None)
)
# i believe if a filter exists then filter is a kwarg
print("Translator output:", _query_text, _kwargs)
print("\nChroma accepts it -- results:")
for d in vectorstore.similarity_search(_query_text, k=5, **_kwargs):
    print(f"  {d.metadata} | {d.page_content[:55]}...")

Translator output: policies {'filter': {'$and': [{'department': {'$eq': 'IT'}}, {'priority': {'$eq': 'critical'}}, {'year': {'$eq': 2025}}]}}

Chroma accepts it -- results:
  {'priority': 'critical', 'department': 'IT', 'year': 2025} | Data Security Policy: All customer data must be encrypt...
  {'department': 'IT', 'priority': 'critical', 'year': 2025} | Password Policy: Passwords must be at least 14 characte...


In [12]:
_kwargs

{'filter': {'$and': [{'department': {'$eq': 'IT'}},
   {'priority': {'$eq': 'critical'}},
   {'year': {'$eq': 2025}}]}}

Confirmed: the hand-built translator produces a real Chroma `where` clause (`{"$and": [{"department": {"$eq": "IT"}}, ...]}`) and Chroma accepts and correctly filters on it — both Data Security Policy and Password Policy come back, both genuinely `department=IT, priority=critical, year=2025`. This proves the translator plumbing works *before* trusting an LLM to be the one building the tree.

Now wire it into an actual `SelfQueryRetriever`, whose job is producing that same tree from natural language.

In [ ]:
metadata_field_info = [
    AttributeInfo(
        name="department",
        description="Which department owns the policy. One of: HR, IT, Finance, Legal.",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the policy was last updated.",
        type="integer",
    ),
    AttributeInfo(
        name="priority",
        description="How critical the policy is. One of: standard, critical.",
        type="string",
    ),
]
document_contents = "A company policy document"

# what do all of these parameters mean
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=get_llm(),  
    vectorstore=vectorstore,
    document_contents=document_contents,
    metadata_field_info=metadata_field_info,
    structured_query_translator=SimpleChromaTranslator(),
    enable_limit=True,
)
print("SelfQueryRetriever constructed -- from_llm() only builds a Runnable chain, no API call yet.")

SelfQueryRetriever constructed -- from_llm() only builds a Runnable chain, no API call yet.


**⚠️ Needs your API key to run — this is where the LLM actually parses your question into a structured query for the first time.**

In [14]:
# Inspect the parsed structured query directly, before it gets translated
parsed = self_query_retriever.query_constructor.invoke({"query": "What are the critical IT policies?"})
print("Parsed structured query:", parsed)

results = self_query_retriever.invoke("What are the critical IT policies?")
print(f"\nSelfQueryRetriever returned {len(results)} documents:")
for d in results:
    print(f"  {d.metadata} | {d.page_content[:60]}...")

Parsed structured query: query=' ' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='department', value='IT'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='priority', value='critical')]) limit=None

SelfQueryRetriever returned 2 documents:
  {'year': 2025, 'department': 'IT', 'priority': 'critical'} | Password Policy: Passwords must be at least 14 characters an...
  {'year': 2025, 'priority': 'critical', 'department': 'IT'} | Data Security Policy: All customer data must be encrypted at...


In [17]:
print(parsed)
print(parsed.filter)

query=' ' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='department', value='IT'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='priority', value='critical')]) limit=None
operator=<Operator.AND: 'and'> arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='department', value='IT'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='priority', value='critical')]


### 🔍 Under the hood — the full path from a sentence to a Chroma filter

One `.invoke("What are the critical IT policies?")` call:
1. **Query constructor chain** (`self_query_retriever.query_constructor`) — an LLM call using a prompt built from `document_contents` + `metadata_field_info`, instructed to output a mini query language expressing `{"query": <semantic part>, "filter": <expression>}`. This is the LLM call in the cell above.
2. **Grammar parsing** — the filter expression comes back as text (e.g. `and(eq("department", "IT"), eq("priority", "critical"))`); `langchain_classic` parses this with a `lark` grammar into real `Operation`/`Comparison` objects (the exact classes `SimpleChromaTranslator` above already knows how to walk).
3. **Translation** — `structured_query_translator.visit_structured_query(...)` walks that tree; here that's `SimpleChromaTranslator`, turning it into the same `{"$and": [...]}`  dict proven in the cell above.
4. **Retrieval** — `vectorstore.similarity_search(semantic_query, filter=where_dict)` runs, restricted to only the matching metadata, same mechanism as `vector_databases_for_rag` Section 5's hand-written `filter=`.

The entire value of `SelfQueryRetriever` is step 1 and 2 — turning free text into that `Operation`/`Comparison` tree. Everything after that (step 3, 4) is exactly the metadata filtering you already know.

### 🔗 Ties back to theory — the concrete answer to "who writes the filter dict?"

`vector_databases_for_rag` Section 5 left the `filter={"category": "landmarks"}` dict as something *you* write, because that notebook was demonstrating the filtering mechanism itself. `SelfQueryRetriever` is the production answer to "the user isn't going to write a filter dict — they're going to type a sentence": the exact same mechanism, just with an LLM as the thing writing the dict instead of you.

## 5. Optional — the raw Anthropic SDK equivalent of both retrievers

Neither `MultiQueryRetriever` nor `SelfQueryRetriever` does anything vendor-specific — both are prompt-and-parse patterns you could write directly against Claude. Seeing the raw version makes clear exactly what each wrapper is buying you.

**⚠️ Both cells below need your API key to run.**

### 5a. Multi-query, raw

In [ ]:
import anthropic

raw_client = anthropic.Anthropic()

# Deliberately reusing MultiQueryRetriever's own default prompt verbatim, so
# this is a fair apples-to-apples comparison, not a weaker hand-rolled version.
rewrite_prompt = f"""You are an AI language model assistant. Your task is
    to generate 3 different versions of the given user
    question to retrieve relevant documents from a vector  database.
    By generating multiple perspectives on the user question,
    your goal is to help the user overcome some of the limitations
    of distance-based similarity search. Provide these alternative
    questions separated by newlines. Original question: {compound_question}"""

response = raw_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[{"role": "user", "content": rewrite_prompt}],
)
rewritten_queries = [line for line in response.content[0].text.strip().split("\n") if line]
print("Generated rephrasings:")
for q in rewritten_queries:
    print(" -", q)

seen = {}
for q in rewritten_queries:
    for d in vectorstore.similarity_search(q, k=2):
        seen[d.page_content] = d

print(f"\nUnion across rephrasings -> {len(seen)} unique documents:")
for d in seen.values():
    print(f"  [{d.metadata['department']}] {d.page_content[:70]}...")

### 5b. Self-query, raw — via forced tool use

In [ ]:
extract_policy_query_tool = {
    "name": "extract_policy_query",
    "description": "Extract a semantic search query and any explicit structured filters from a question about company policies.",
    "input_schema": {
        "type": "object",
        "properties": {
            "semantic_query": {
                "type": "string",
                "description": "The core topic to search for, with filter language stripped out.",
            },
            "department": {
                "type": "string",
                "enum": ["HR", "IT", "Finance", "Legal"],
                "description": "Only set if the question names or clearly implies one specific department.",
            },
            "year": {
                "type": "integer",
                "description": "Only set if the question names a specific year.",
            },
            "priority": {
                "type": "string",
                "enum": ["standard", "critical"],
                "description": "Only set if the question specifically asks about critical/urgent policies.",
            },
        },
        "required": ["semantic_query"],
    },
}

response = raw_client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=300,
    tools=[extract_policy_query_tool],
    tool_choice={"type": "tool", "name": "extract_policy_query"},
    messages=[{"role": "user", "content": "What are the critical IT policies?"}],
)
tool_input = response.content[0].input
print("Extracted:", tool_input)

# Build the same Chroma `where` dict shape by hand, only from keys the model actually set
where_clauses = [
    {key: {"$eq": value}}
    for key, value in tool_input.items()
    if key != "semantic_query"
]
where_filter = {"$and": where_clauses} if len(where_clauses) > 1 else (where_clauses[0] if where_clauses else None)

kwargs = {"filter": where_filter} if where_filter else {}
results = vectorstore.similarity_search(tool_input["semantic_query"], k=5, **kwargs)
print(f"\n{len(results)} result(s):")
for d in results:
    print(f"  {d.metadata} | {d.page_content[:55]}...")

**The mapping, made explicit:**

| LangChain piece | Raw SDK equivalent |
|---|---|
| `MultiQueryRetriever.from_llm(...)` | Formatting the same default prompt by hand, `client.messages.create()` |
| `LineListOutputParser` | `response.content[0].text.strip().split("\n")` |
| Automatic per-rephrasing retrieval + de-dup | The `for q in rewritten_queries: ... seen[d.page_content] = d` loop |
| `SelfQueryRetriever.from_llm(...)` | A forced tool call (`tool_choice`) whose schema mirrors `AttributeInfo` |
| Query-constructor chain + `lark`-parsed filter tree | Not needed — the tool's `input` dict is *already* structured |
| `structured_query_translator.visit_*` | The manual `if key in tool_input: where[...] = ...` dict-building above |

**One genuinely interesting asymmetry:** the raw self-query version is *simpler* than `SelfQueryRetriever`'s own internals — no grammar, no separate query-constructor chain, no translator class needed at all if you're willing to build the filter dict inline like above. That's because Claude's native tool-use already enforces structured output; `SelfQueryRetriever`'s extra machinery (the mini query language, the `lark` grammar, the `Visitor` pattern) exists so the *same* retriever code works across LLMs that don't have reliable native tool-calling. That generality is a real cost you're paying for even when — like here — you only ever target Claude and don't need it.

## 6. Re-ranking — fixing the final ordering, not the candidate pool

A different failure mode from Sections 3-4: the right document is genuinely *in* the retrieved set, just not on top, because it only shares partial surface vocabulary with the query while a less-relevant document happens to share more. No LLM needed for this one — just a second, more precise (and more expensive) model applied to a small candidate pool instead of the whole corpus.

**Bi-encoder vs. cross-encoder, precisely — this is the actual mechanism.** `HuggingFaceEmbeddings` (used everywhere in this notebook so far) is a **bi-encoder**: it encodes the query and each document *independently* into fixed vectors, then compares them with cosine similarity. That independence is exactly what makes it fast at scale — every document's vector gets computed once, ahead of time, and stored; a query only needs one new vector, compared against millions of pre-computed ones. A **cross-encoder** does the opposite: it feeds the query and *one specific document* through a transformer *together*, in the same forward pass, so every token of the query can attend to every token of the document (and vice versa) before producing a single relevance score. That joint attention is far more precise — but it means nothing can be pre-computed; a cross-encoder has to run once per query-document *pair*, at query time. That's why cross-encoders never replace vector search — they can't scale to searching a whole corpus — they only ever re-score a small pool something else already retrieved.

In [3]:
from sentence_transformers import CrossEncoder

# Same library already backing HuggingFaceEmbeddings -- no new dependency, and no
# Anthropic API key needed anywhere in this section, since Claude is never involved.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Deliberately adversarial for a bi-encoder: "products or trips from vendors"
# shares surface vocabulary with the Finance-department policies (expenses,
# travel booking) even though the actually-correct policy is Code of Conduct
# (accepting gifts/incentives from vendors is a conflict-of-interest question).
rerank_question = "rules for accepting free products or trips from vendors we do business with"

plain_results = vectorstore.similarity_search(rerank_question, k=3)
print("Plain similarity_search, k=3:")
for rank, d in enumerate(plain_results, 1):
    print(f"  {rank}. [{d.metadata['department']}] {d.page_content[:60]}...")

# CrossEncoder.predict wants (query, document_text) pairs, scored one at a time --
# this is the "run once per pair, at query time" cost described above, only
# affordable because plain_results has already narrowed the field to 3 documents.
pairs = [(rerank_question, d.page_content) for d in plain_results]
scores = reranker.predict(pairs)
reranked_results = sorted(zip(plain_results, scores), key=lambda pair: -pair[1])

print("\nAfter cross-encoder re-ranking:")
for rank, (d, score) in enumerate(reranked_results, 1):
    print(f"  {rank}. [{score:.2f}] [{d.metadata['department']}] {d.page_content[:60]}...")

Plain similarity_search, k=3:
  1. [Finance] Travel and Reimbursement Policy: Employees traveling for bus...
  2. [Finance] Expense Policy: Business expenses under $50 do not require p...
  3. [Legal] Code of Conduct: Employees must avoid conflicts of interest ...



After cross-encoder re-ranking:
  1. [-6.14] [Finance] Travel and Reimbursement Policy: Employees traveling for bus...
  2. [-10.14] [Legal] Code of Conduct: Employees must avoid conflicts of interest ...
  3. [-10.33] [Finance] Expense Policy: Business expenses under $50 do not require p...


**Honest reading of the real result above — this is a more modest demonstration than Section 2's, and worth saying so directly.** Plain search puts Code of Conduct — the genuinely correct policy for "accepting gifts from a vendor" — at position 3, behind Travel and Reimbursement and Expense Policy, both of which only match on the surface words "trips"/"products"/"business." Re-ranking correctly promotes Code of Conduct to position 2, fixing a real ordering mistake. It does *not* flip who's ranked first here; across many test queries on this small, clean, 10-document corpus, a dramatic top-1 rescue (the kind Section 2 got from the compound question) turned out to be genuinely hard to produce — the bi-encoder is already good enough at top-1 on a corpus this size and this cleanly separated by topic. That's not a weakness in re-ranking; it's the same honesty Section 2 already showed in the other direction — re-ranking's value is real but shows up more reliably at larger scale, with more documents that genuinely compete on surface vocabulary, than a 10-document teaching corpus can dramatize.

### 🔗 Ties back to theory — this is `vector_databases_for_rag` §9's exercise 5, actually done

That notebook introduced hybrid search (BM25 + vector) and named re-ranking as the natural next step — *"install `sentence-transformers` cross-encoder models... and use one to re-score the top 10 hybrid search results"* — but left it as an unaddressed exercise. `reranker.predict(pairs)` above is exactly that model family (`cross-encoder/ms-marco-MiniLM-L-6-v2`), applied to a plain-search candidate pool instead of a hybrid one — the same re-scoring mechanism would apply just as well on top of `vector_databases_for_rag`'s BM25+vector union, or on top of `MultiQueryRetriever`'s wider pool from Section 3, if you wanted to combine techniques (Exercise 6 below).

## 7. When to actually reach for these

Neither retriever is free:

- **`MultiQueryRetriever`** costs 1 extra LLM call plus up to *N×* the retrieval calls (3 rephrasings × k here = up to 6 retrievals for one user question). Worth it when questions are genuinely compound, or when your users' vocabulary is known to diverge from your source documents' — not a blanket "better retrieval" switch.
- **`SelfQueryRetriever`** costs 1 extra LLM call for parsing, and needs accurate `AttributeInfo` descriptions maintained as your metadata schema evolves. Worth it specifically when users naturally phrase both a topic *and* a filter in one sentence.
- **Re-ranking** (§6) costs one cross-encoder pass over the whole candidate pool, no LLM call at all — cheaper in dollars than either retriever above, but slower per-call than plain vector search (no pre-computation possible) and doesn't help if the truly relevant document was never retrieved into the pool in the first place. Complementary to, not a replacement for, `MultiQueryRetriever`/`SelfQueryRetriever`.

Neither claim above should be taken on faith, including the "genuinely helps" claim demonstrated in Section 2/3 — that's exactly what Module 04 (Evaluation for LLM Apps) is for: run `vector_databases_for_rag` Section 8's precision@k / recall@k / MRR harness against plain retrieval, `MultiQueryRetriever`, and `SelfQueryRetriever` on the same labeled question set, and see which one actually moves the numbers for *your* queries — "advanced" is not the same claim as "better," and the only way to tell them apart is to measure.

**A currency note, same shape as 02's `create_supervisor` finding.** Checked directly against docs.langchain.com: both classes are still real and correctly imported — the official `langchain-classic` migration guide names `MultiQueryRetriever` *specifically* as an example of what moved there, which is about as solid a confirmation as this gets. But current top-level retrieval guidance now organizes around "2-Step / Agentic / Hybrid RAG" architectures, and describes multi-query specifically as a **step you build into a custom LangGraph workflow** ("query enhancement... generating multiple variations") rather than leading with a standalone retriever class to import. Nothing here is wrong or needs fixing — both retrievers work exactly as built above — but if you go looking at current docs directly, don't be surprised that `MultiQueryRetriever`/`SelfQueryRetriever` aren't the first thing you find; the same underlying idea is increasingly framed as something you hand-roll as a graph node, same trend as `langgraph-supervisor` in 02.

## Exercises

1. **Add a new metadata field.** Add `"policy_owner"` (a person's name) to a few documents' metadata, extend `metadata_field_info` with a matching `AttributeInfo`, and confirm `SelfQueryRetriever` can filter on it from a question like *"Which policies does Priya own?"*
2. **Try `include_original=True`.** Pass it to `MultiQueryRetriever.from_llm(...)` and compare the returned document count against Section 3's run — the original phrasing becomes a 4th query alongside the 3 generated rephrasings.
3. **Combine both retrievers.** Build a `MultiQueryRetriever` whose `retriever=` argument is `vectorstore.as_retriever(search_kwargs={"filter": {"department": "IT"}})` instead of the plain one from Section 3 — now compound questions get rephrased *and* department-scoped at the same time.
4. **Break `SimpleChromaTranslator` on purpose.** Remove `Comparator.IN` from `allowed_comparators` and ask a question like *"Show me HR or IT policies"* — read the exact error `SelfQueryRetriever` raises when the LLM tries to use a disallowed comparator.
5. **Actually measure it.** Build a small labeled eval set (question → expected `doc_id` set, same shape as `vector_databases_for_rag` Section 8) covering both compound and filter-heavy questions, and compute precision@k/recall@k/MRR for plain search vs. `MultiQueryRetriever` vs. `SelfQueryRetriever` side by side.
6. **Chain re-ranking onto a wider pool.** Take `MultiQueryRetriever`'s 6-document result from Section 3 (the compound-question run) and re-rank it with Section 6's `reranker` against the original `compound_question` — does the Expense Policy (recovered by `MultiQueryRetriever` but never separately verified as well-ranked) actually land near the top once re-ranked, or does it stay buried?
7. **Try to find a bigger top-1 flip than Section 6 did.** Section 6's demo only ever moved the correct document from position 3 to position 2, never to position 1 — write 10-15 more adversarial queries against this same 10-document corpus and see if you can construct a case where re-ranking changes the *top* result, not just the runner-up. If you can't, that's itself a real finding worth writing down: at what corpus size/ambiguity does re-ranking start changing top-1 answers, not just close ones?